In [1]:
import pandas as pd

# Load files
private_raw = pd.read_excel('datasets/private.xlsx')
motorcycle_raw = pd.read_excel('datasets/motorcycle.xlsx')
light_raw = pd.read_excel('datasets/light.xlsx')
heavy_raw = pd.read_excel('datasets/heavy.xlsx')
toyota_raw = pd.read_excel('datasets/toyota.xlsx')

In [ ]:
#### THIS IS OLD, DO NOT RUN
# clean data
private = private_raw.drop(columns=list(private_raw.columns[-1:]) + ['IMPORT STATUS (SHORT)', 'NEW PRICE', 'UOM (UNIT OF MEASUREMENT']).rename(columns=str.lower)
motorcycle = motorcycle_raw.drop(columns=list(motorcycle_raw.columns[-5:]) + ['IMPORT STATUS (SHORT)', 'NEW PRICE', 'UNIT OF MEASUREMENT(UOM)']).rename(columns=str.lower)
light = light_raw.drop(columns=list(light_raw.columns[-2:]) + ['Description', 'Import Status.1', 'Body Type in BM']).rename(columns=str.lower)
heavy = heavy_raw.drop(columns=list(heavy_raw.columns[-3:-1]) + ['Production Start and End Year', 'Import Status.1', 'Body Type in BM', 'West New Price']).rename(columns=str.lower)
toyota = toyota_raw.drop(columns=list(toyota_raw.columns[-7:]) + ['PENINSULAR MALAYSIA'], index=0).rename(columns=str.lower)

# reorder columns
cols = list(light.columns)
a, b = cols.index('import status'), cols.index('year')
cols[a], cols[b] = cols[b], cols[a]
light = light[cols]

cols = list(heavy.columns)
a, b = cols.index('import status'), cols.index('year')
cols[a], cols[b] = cols[b], cols[a]
heavy = heavy[cols]


light.insert(loc=0, column="series", value=None)

# rename columns
toyota.rename(columns={'unnamed: 11': 'value', 'family': 'model'}, inplace=True)
private.rename(columns={'capacity': 'cc', 'quater 1 jan 2026 snk medium west value\n(round up)': 'value', 'family': 'model'}, inplace=True)
motorcycle.rename(columns={'capacity': 'cc', 'q1 jan 2026 west value': 'value', 'family': 'model'}, inplace=True)
light.rename(columns={' west mv july 2025 ': 'value'}, inplace=True)
heavy.rename(columns={'west mv jan 2026': 'value'}, inplace=True)

# combine the toyota and private dataframes
new_private = pd.concat([toyota, private], ignore_index=True)
new_private = new_private.assign(value=new_private.pop('value'))

In [196]:
from numpy import nan
from uuid import uuid5, NAMESPACE_OID

private = private_raw.rename(columns=str.lower).drop(
    columns=['uom (unit of measurement',
            'import status'] + list(private_raw.columns[-1:].str.lower())).rename(
    columns={'quater 1 jan 2026 snk medium west value\n(round up)': 'value',
            'import status (short)': 'import_status',
            'capacity': 'cc',
            'family': 'model'}).replace({'RECOND': 'RCN'})

toyota = toyota_raw.rename(columns=str.lower).drop(
    columns=['peninsular malaysia'] + list(toyota_raw.columns[-7:].str.lower()), index=0).rename(
    columns={'unnamed: 11': 'value',
            'family': 'model'})
toyota.insert(loc=0, column="import_status", value=None)

motorcycle = motorcycle_raw.rename(columns=str.lower).drop(
    columns=['unit of measurement(uom)', 'import status'] + list(motorcycle_raw.columns[-5:].str.lower())).rename(
    columns={'family': 'model',
            'capacity': 'cc',
            'import status (short)': 'import_status',
            'q1 jan 2026 west value': 'value'})

light = light_raw.rename(columns=str.lower).drop(
    columns=['body type in bm', 'import status'] + list(light_raw.columns[-2:].str.lower())).rename(
    columns={'body type in english ': 'style',
            'import status.1': 'import_status',
            ' west mv july 2025 ': 'value'})
light.insert(loc=0, column="series", value=None)

heavy = heavy_raw.rename(columns=str.lower).drop(
    columns=['body type in bm', 'import status'] + list(heavy_raw.columns[-3:-2].str.lower())).rename(
    columns={'body type in english': 'style',
            'import status.1': 'import_status',
            'west mv jan 2026': 'value'})

columns = ['category', 'make', 'model', 'variant', 'series', 'year',
           'cc', 'import_status', 'transmission', 'style',
           'value', 'specs']

def organize(df):
    sample = df.copy()

    specs_col = df.columns.difference(columns)
    sample['specs'] = sample[specs_col].apply(lambda row: row.dropna().to_dict(), axis=1)
    sample['id'] = sample[columns[1:-2]].apply(lambda row: str(uuid5(NAMESPACE_OID, ','.join(row.dropna().astype(str).to_list()))), axis=1)
    
    sample.drop(specs_col, axis=1, inplace=True)
    sample.drop_duplicates('id', inplace=True)
    return sample.astype({'year': int, 'cc': int, 'value': float})

vehicles = pd.concat([
    organize(private),
    organize(toyota),
    organize(motorcycle),
    organize(light),
    organize(heavy),
], keys=['private', 'private', 'motorcycle', 'light', 'heavy']).reset_index().rename(
    columns={'level_0': 'category'}).drop('level_1', axis=1).replace({nan: None})

In [228]:
# str(uuid5(NAMESPACE_OID, ','.join(vehicles.iloc[23542][columns[1:-2]].dropna().astype(str).to_list())))
# vehicles.iloc[23542]
# ','.join(vehicles.iloc[23542][columns[1:-2]].dropna().astype(str).to_list())
NAMESPACE_OID

UUID('6ba7b812-9dad-11d1-80b4-00c04fd430c8')

In [ ]:
#### THIS IS OLD, DO NOT RUN
from numpy import nan
columns = ['category', 'make', 'model', 'variant', 'series', 'year', 'cc', 'import status', 'transmission', 'style', 'value', 'specs']

table_map = {
    'private': new_private,
    'motorcycle': motorcycle,
    'light': light,
    'heavy': heavy,
}

def classify(category: str, row: pd.Series):
    to_str = lambda x: str(x) if x is not None else None
    return (
        category,
        to_str(row.get('make')),
        to_str(row.get('model')),
        to_str(row.get('variant')),
        to_str(row.get('series')),
        int(row.get('year')) if row.get('year') is not None else None,
        row.iloc[5:-1].dropna().to_dict(),
        float(row.get('value'))
    )

def classify_all_tables():
    unique = []
    dupelicates = 0 

    for k, v in table_map.items():
        print('Starting ' + k, end=' ')

        for i, row in v.iterrows():
            classification = classify(k, row)
            if classification not in unique:
                unique.append(classification)
            else: dupelicates+=1

        print('✅')

    print(str(dupelicates) + " Dupelicates!")
    output = pd.DataFrame(unique, columns=columns).replace({nan: None})
    output['year'] = output['year'].astype(int) # fix year datatype, sometimes could be float
    return output

# file_vehicles = classify_all_tables()

In [3]:
import os
from supabase import create_client, Client
from dotenv import load_dotenv
load_dotenv('.env.local')

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_SERVICE_ROLE_KEY")

supabase: Client = create_client(url, key)

In [198]:
""" # Supabase Constraint
ALTER TABLE vehicles
ADD CONSTRAINT unique_vehicle_identity 
UNIQUE NULLS NOT DISTINCT (category, year, make, model, variant, series, style, transmission, cc, import_status);
"""

# Uploading data to table
# constraints = ', '.join([columns[1:-2]])
for i in range(0, len(vehicles), 1000):
    print(f'Uploading batch {i}-{i+1000} out of {len(vehicles)}', end=" ")
    batch = vehicles.iloc[i:i+1000, vehicles.columns != 'value'].to_dict(orient='records')
    supabase.table('vehicles').upsert(
        batch, on_conflict='id'
        ).execute()
    print('✅')

Uploading batch 0-1000 out of 59202 ✅
Uploading batch 1000-2000 out of 59202 ✅
Uploading batch 2000-3000 out of 59202 ✅
Uploading batch 3000-4000 out of 59202 ✅
Uploading batch 4000-5000 out of 59202 ✅
Uploading batch 5000-6000 out of 59202 ✅
Uploading batch 6000-7000 out of 59202 ✅
Uploading batch 7000-8000 out of 59202 ✅
Uploading batch 8000-9000 out of 59202 ✅
Uploading batch 9000-10000 out of 59202 ✅
Uploading batch 10000-11000 out of 59202 ✅
Uploading batch 11000-12000 out of 59202 ✅
Uploading batch 12000-13000 out of 59202 ✅
Uploading batch 13000-14000 out of 59202 ✅
Uploading batch 14000-15000 out of 59202 ✅
Uploading batch 15000-16000 out of 59202 ✅
Uploading batch 16000-17000 out of 59202 ✅
Uploading batch 17000-18000 out of 59202 ✅
Uploading batch 18000-19000 out of 59202 ✅
Uploading batch 19000-20000 out of 59202 ✅
Uploading batch 20000-21000 out of 59202 ✅
Uploading batch 21000-22000 out of 59202 ✅
Uploading batch 22000-23000 out of 59202 ✅
Uploading batch 23000-24000 out o

In [197]:
# checking for deuplicates
df = vehicles.copy()
conflicts = columns[1:-2]
df['specs'] = df['specs'].astype(str)

dupe_mask = df.duplicated(subset='id', keep=False)
dupelicates = df[dupe_mask].sort_values(by=['make', 'model', 'year'])
dupelicates.to_csv('dupelicates.csv', index=False)
# conflicts

In [ ]:
# NOT USED ANYMORE BECAUSE OF HASHING ID BEFORE UPLOADING
# re-fetch vehicles to build a new map with uuid
NO_VEHICLES = supabase.table('vehicles').select("*", count='exact').limit(1).execute()
SELECT_QUERY = supabase.table('vehicles').select("*").limit(1000)

batches = []
for offset in range(0, NO_VEHICLES.count, 1000):
    print(f'Downloading batch {offset}-{offset+1000} out of {NO_VEHICLES.count}', end=" ")
    new_vehicles = SELECT_QUERY.offset(offset).execute()
    batches += new_vehicles.data
    print('✅')

live_vehicles = pd.DataFrame(batches, columns=list(NO_VEHICLES.data[0].keys()))
# live_vehicles = vehicles
# combined = test.merge(vehicles, on=test.columns[1:-1].to_list(), how='left')
# combined.rename({'specs_x': 'specs'}, inplace=True, axis=1)
# combined.drop('specs_y', inplace=True, axis=1)
# # combined[combined['id'].isna()]
# combined

In [227]:
# Add to vehicle value table

date = '2026-jan-01'
batch_size = 1000

for i in range(2000, len(vehicles), batch_size):
    print(f'Uploading batch {i}-{i+batch_size} out of {len(vehicles)}', end=" ")
    batch = vehicles.iloc[i:i+batch_size][['id', 'value']].rename({'id': 'vehicle_id'}, axis=1)
    batch.insert(0, 'valuation_period', date)
    supabase.table('vehicle_values').upsert(
        batch.to_dict(orient='records'), on_conflict='id'
        ).execute()
    print('✅')
    # print(batch)
    # break

Uploading batch 2000-3000 out of 59202 ✅
Uploading batch 3000-4000 out of 59202 ✅
Uploading batch 4000-5000 out of 59202 ✅
Uploading batch 5000-6000 out of 59202 ✅
Uploading batch 6000-7000 out of 59202 ✅
Uploading batch 7000-8000 out of 59202 ✅
Uploading batch 8000-9000 out of 59202 ✅
Uploading batch 9000-10000 out of 59202 ✅
Uploading batch 10000-11000 out of 59202 ✅
Uploading batch 11000-12000 out of 59202 ✅
Uploading batch 12000-13000 out of 59202 ✅
Uploading batch 13000-14000 out of 59202 ✅
Uploading batch 14000-15000 out of 59202 ✅
Uploading batch 15000-16000 out of 59202 ✅
Uploading batch 16000-17000 out of 59202 ✅
Uploading batch 17000-18000 out of 59202 ✅
Uploading batch 18000-19000 out of 59202 ✅
Uploading batch 19000-20000 out of 59202 ✅
Uploading batch 20000-21000 out of 59202 ✅
Uploading batch 21000-22000 out of 59202 ✅
Uploading batch 22000-23000 out of 59202 ✅
Uploading batch 23000-24000 out of 59202 ✅
Uploading batch 24000-25000 out of 59202 ✅
Uploading batch 25000-2600

In [200]:
vehicles.head()

,category,year,make,model,variant,series,style,transmission,cc,import_status,value,specs,id
0,private,1995,ALFA ROMEO,145,None,None,2D HATCHBACK,5 SP MANUAL,1598,CBU,3000.0,"{'engine': 'MULTI POINT F/INJ', 'engine type':...",9988e88c-227e-5d10-bb49-0105d6a94afe
1,private,1996,ALFA ROMEO,145,None,None,2D HATCHBACK,5 SP MANUAL,1598,CBU,4000.0,"{'engine': 'MULTI POINT F/INJ', 'engine type':...",aef1bef9-7d5d-5865-b479-614a572c8b4a
2,private,1997,ALFA ROMEO,145,None,None,2D HATCHBACK,5 SP MANUAL,1598,CBU,4000.0,"{'engine': 'MULTI POINT F/INJ', 'engine type':...",605137af-3b48-51cb-b1b8-05c8c263c1b6
3,private,1998,ALFA ROMEO,145,None,None,2D HATCHBACK,5 SP MANUAL,1598,CBU,4000.0,"{'engine': 'MULTI POINT F/INJ', 'engine type':...",74f1b8a4-bbdd-5598-bf3d-adafd1d0b05c
4,private,1999,ALFA ROMEO,145,None,None,2D HATCHBACK,5 SP MANUAL,1598,CBU,5000.0,"{'engine': 'MULTI POINT F/INJ', 'engine type':...",7747f853-a282-5528-bfed-abbb1f721a99
